<a href="https://colab.research.google.com/github/1900690/depth-estimation/blob/main/depth_pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Depth pro で深度推定

In [1]:
import os

try:
    # 既にインストール済みかチェック
    import depth_pro
    import numpy as np
    # NumPyの互換性チェック（エラーが出ればexceptへ）
    print(f"Checking environment... NumPy {np.__version__}")
except (ImportError, ValueError):
    print("Environment setup required. Installing and restarting...")
    # 1. 依存ライブラリのインストール
    # %%capture を使うとログを隠せますが、再起動を優先するためOSコマンドで実行
    os.system("pip install --upgrade numpy")
    os.system("pip install git+https://github.com/apple/ml-depth-pro.git")

    # 2. モデルのダウンロード
    os.makedirs("checkpoints", exist_ok=True)
    os.system('wget -q -L "https://huggingface.co/apple/DepthPro/resolve/main/depth_pro.pt?download=true" -O checkpoints/depth_pro.pt')

    # 3. ランタイムを強制終了（Colabが自動で再起動します）
    print("!!! RESTARTING RUNTIME !!! Please wait a moment...")
    os._exit(0)

# --- 再起動後、ここから下の処理が実行されます ---
import torch
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, transform = depth_pro.create_model_and_transforms()
model = model.to(device).eval()

print(f"Success! Model loaded on {device}.")

Checking environment... NumPy 1.26.4
Success! Model loaded on cpu.


In [ ]:
import torch
import depth_pro
import numpy as np
import cv2
import PIL.Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg

# --- 設定パラメータ ---
DIM = (1280, 720) # オリジナルのサイズ

def create_depth_visualization(depth, width, height, v_max_limit, use_limit):
    """
    matplotlibを使用して、タイトルとカラーバー付きの深度マップ画像を作成
    ※この時点のheightは切り取り後の高さ
    """
    fig, ax = plt.subplots(figsize=(width/100, height/100), dpi=100)

    v_min = depth.min()
    v_max = v_max_limit if use_limit else depth.max()

    # 深度マップの描画
    im = ax.imshow(depth, cmap='rainbow', vmin=v_min, vmax=v_max)

    title = f"Min: {v_min:.1f}m, Max: {v_max:.1f}m"
    if use_limit: title += " (Limit Fixed)"
    ax.set_title(f"Predicted Depth Map - {title}", fontsize=15)
    ax.axis('off')

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Depth [m]', rotation=270, labelpad=15)

    canvas = FigureCanvasAgg(fig)
    canvas.draw()
    img_viz = cv2.cvtColor(np.asarray(canvas.buffer_rgba()), cv2.COLOR_RGBA2BGR)
    plt.close(fig)

    # 指定されたサイズにリサイズして返す
    return cv2.resize(img_viz, (width, height))

def run_depth_estimation_final(input_path, output_path,
                               cam_height=16.5,
                               use_cam_limit=True,
                               use_ref_correction=False, # 補正を使用するか
                               ref_point=None,           # (x, y) 座標
                               ref_distance=None,        # 実際の距離(m)
                               mask_height_px=60,        # 切り取る高さ
                               frame_interval=5):

    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print(f"エラー: {input_path} が見つかりません。")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # 出力される映像の高さ（マスク分を引く）
    output_h = DIM[1] - mask_height_px
    output_w = DIM[0]

    # VideoWriterの設定 (横幅は2倍：元画像 + 深度マップ)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (output_w * 2, output_h))

    pbar = tqdm(total=total_frames, desc="Processing")
    frame_idx = 0
    middle_frame_img = None

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        if frame_idx % frame_interval == 0:
            # 1. 魚眼補正
            undistorted = cv2.remap(frame, map1, map2, interpolation=cv2.INTER_LINEAR)

            # 2. 推論用の画像準備（AIには一旦全画面見せるが、日付部分は黒塗りして影響を排除）
            img_inference = cv2.cvtColor(undistorted, cv2.COLOR_BGR2RGB)
            if mask_height_px > 0:
                img_inference[-mask_height_px:, :] = 0

            pil_img = PIL.Image.fromarray(img_inference)
            input_data = transform(pil_img)

            if torch.cuda.is_available():
                input_data = input_data.cuda()

            with torch.no_grad():
                prediction = model.infer(input_data)
                depth = prediction["depth"].cpu().numpy()

            # 3. 深度補正 (オンの場合のみ実行)
            if use_ref_correction and ref_point is not None and ref_distance is not None:
                px_x, px_y = ref_point
                # 指定座標の現在の予測値
                predicted_val = depth[px_y, px_x]
                if predicted_val > 0:
                    scale_factor = ref_distance / predicted_val
                    depth = depth * scale_factor

            # 4. 切り取り（日付部分をカット）
            # 元画像と深度データの両方を同じ高さでクロップ
            undistorted_cropped = undistorted[:output_h, :]
            depth_cropped = depth[:output_h, :]

            # 5. 可視化作成
            depth_viz = create_depth_visualization(depth_cropped, output_w, output_h, cam_height, use_cam_limit)

            # 6. 左右に結合
            combined = np.hstack((undistorted_cropped, depth_viz))

            # 補正点のマーカー（確認用：切り取り範囲内の場合のみ描画）
            if use_ref_correction and ref_point:
                if ref_point[1] < output_h:
                    cv2.circle(combined, ref_point, 7, (0, 255, 0), -1)

            out.write(combined)

            # 中間フレーム保持用
            if frame_idx >= (total_frames // 2) and middle_frame_img is None:
                middle_frame_img = combined.copy()

        frame_idx += 1
        pbar.update(1)

    cap.release()
    out.release()

    # 中間フレームの表示
    if middle_frame_img is not None:
        plt.figure(figsize=(15, 8))
        plt.imshow(cv2.cvtColor(middle_frame_img, cv2.COLOR_BGR2RGB))
        plt.title("Analysis Result (Middle Frame)")
        plt.axis('off')
        plt.show()

# --- 実行設定 ---

config = {
    "cam_height": 16.5,            # カラーバーの最大値（上限）
    "use_cam_limit": True,         # cam_heightを上限として固定するか
    "use_ref_correction": True,    # 補正機能をオンにするか
    "ref_point": (640, 400),       # 距離が既知のポイント座標 (x, y)
    "ref_distance": 10.0,          # そのポイントの実際の距離 (m)
    "mask_height_px": 70,          # 日付部分を切り取る高さ（ピクセル）
    "frame_interval": 5
}

run_depth_estimation_final("input.mp4", "output_final_cropped.mp4", **config)